In [1]:
import os

In [2]:
import torch

In [3]:
from torch import nn

In [4]:
from torch.utils.data import DataLoader

In [5]:
from torchvision import datasets, transforms

In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [7]:
print(f"using {device} device")

using cuda device


In [8]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [9]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [12]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([8], device='cuda:0')


In [13]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [14]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [15]:
layer1 = nn.Linear(in_features = 28*28, out_features = 20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [16]:
print(f"Before Relu: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before Relu: tensor([[ 0.1643, -0.2458, -0.3499,  0.0644, -0.0828,  0.2549, -0.3892,  0.5530,
         -0.0711, -0.4226, -0.1205,  0.5182, -0.2154, -0.1360,  0.2466, -0.0915,
         -0.0282,  0.0496, -0.3386, -0.5314],
        [ 0.2174, -0.5448, -0.0339, -0.0545, -0.1589,  0.7183, -0.2100,  0.3958,
          0.0432, -0.2641, -0.0930,  0.1494, -0.1860,  0.0104, -0.0657, -0.0197,
          0.0559, -0.0296, -0.3184, -0.0283],
        [-0.0312, -0.5019, -0.1614, -0.1568, -0.1844,  0.2969, -0.2171,  0.3948,
         -0.0949, -0.3595, -0.1717,  0.0373,  0.2971,  0.2750,  0.1859,  0.3057,
          0.1362,  0.1400, -0.3340, -0.2949]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.1643, 0.0000, 0.0000, 0.0644, 0.0000, 0.2549, 0.0000, 0.5530, 0.0000,
         0.0000, 0.0000, 0.5182, 0.0000, 0.0000, 0.2466, 0.0000, 0.0000, 0.0496,
         0.0000, 0.0000],
        [0.2174, 0.0000, 0.0000, 0.0000, 0.0000, 0.7183, 0.0000, 0.3958, 0.0432,
         0.0000, 0.0000, 0.1494, 0.0000, 0.0104, 0.00

In [17]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20,10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [19]:
softmax = nn.Softmax(dim = 1)
pred_probab = softmax(logits)

In [20]:
print(f"Model structure: {model}\n\n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)




In [22]:
for name, param in model.named_parameters():
    print(f"Layer: {name} | size: {param.size()} | values: {param[:2]} \n")

Layer: linear_relu_stack.0.weight | size: torch.Size([512, 784]) | values: tensor([[-0.0161, -0.0006, -0.0276,  ..., -0.0299, -0.0043,  0.0136],
        [-0.0281, -0.0045,  0.0024,  ...,  0.0237,  0.0279, -0.0037]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | size: torch.Size([512]) | values: tensor([0.0220, 0.0184], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | size: torch.Size([512, 512]) | values: tensor([[-0.0308, -0.0124,  0.0024,  ..., -0.0008, -0.0162,  0.0439],
        [ 0.0154,  0.0106,  0.0428,  ...,  0.0102,  0.0320, -0.0355]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | size: torch.Size([512]) | values: tensor([-0.0431,  0.0062], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.4.weight | size: torch.Size([10, 512]) | values: tensor([[ 0.0415,  0.0072,  0.0433,  ..., -0.0290,  0.0098,  0.0431],
        [ 0.0419, -0.0210,  0.0219,  ..., 